# A3 Multi Recipe Nested LOSO

Exploratory six-recipe A3/TCN/GRU comparison. 648 inner fits + 36 refits; 12 real GPU smoke fits outside this budget. Fixed threshold 0.5. Three seeds, four outer subjects. No test-driven changes; no guarantee of improvement.

Use GPU T4 x2, Internet on, Save & Run All for background execution. Source is pinned and SHA-verified. No package installs or paid resources. Partial metrics are not final results.


In [ ]:
import hashlib, json, os, sys, time, urllib.request, platform
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
import torch
torch.set_num_threads(2)
assert torch.cuda.device_count() == 2, 'This registration requires two GPUs'
assert len(set(torch.cuda.get_device_name(i) for i in range(2))) == 1, 'Match GPU types'
SOURCE_COMMIT = '7543cac24fcb1e5ba05554d4aa7113b29ad3632d'
SOURCE_MANIFEST = {'docs/research/a3-tuning-execution.md': 'c0f7fb16c67a3f4c360a3c60215ff4822effa5a4590343190c61d17169641ec0', 'docs/research/a3-tuning-plan.md': 'a014ada6a7083b049f2692a4d3d6bf5ae79aef1a6dce778c848359d8a1ab1a49', 'docs/research/gru-ctrgcn-addendum.md': '57f40171e0cdcaf8424e879fef5ec2c40d4c42a6e851f59224e9e02b4dc1d978', 'results/component_ablation/locked_protocol.json': 'ce07d98c34f661c3605455b77cb51670700565075a6c540cf1076cf6cdb20b88', 'scripts/training_inputs.py': '68582f1f310b2bbde0f698b1b6d78f33f92243c53028c2a092bc1d70ae99f4ed', 'training/a3_tuning/__init__.py': '01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b', 'training/a3_tuning/runner.py': 'a17c6e08f3cf0c0708ef9b12262148b4cd4014bceb07194417147092917082e5', 'training/a3_tuning/test_runner.py': 'bc38518960b30784ed8ea2d951d78f05a011b1e53ff372c3b5efca2a9d2b0c0e', 'training/backbone_benchmark/models.py': 'a3ccec6a61e04da23c1269b01013e1684375f12c80c9f1aaf0d062b7a8c6c7ad', 'training/backbone_benchmark/runner.py': '8f9d4f5b5ec77b8009259e802e25ddee1254871051b2ee9987053bfec8d8ed62', 'training/backbone_benchmark/supplement/models.py': '8cd08ecb965ef384d7d63d72e8a7aa244c2371b4c5e251be416476b6784e3cf6', 'training/backbone_benchmark/supplement/runner.py': 'bee04449a6de54fa5612fd1c20559536c054abd0d0065a065e7dffa76a28b17c', 'training/backbone_benchmark/supplement/test_supplement.py': '5146fd0f63226e1db1c1f042b5bbd1157256fb4d807db73c44135425a7c8a37c', 'training/backbone_benchmark/supplement/tools/audit.py': '271c5fed30d43e210360241a90e4f782bc73a9f71a8ce14e9e395299179a04d4', 'training/backbone_benchmark/supplement/tools/monitor_windows.py': 'fbb5200d99ddc5232a955b55dcd256fc6895555cd41698195d0d379e9d928b39', 'training/backbone_benchmark/supplement/tools/test_monitor_windows.py': '2e053af7c9d41144b43590f087e9c1ab3363c1afaaadf9d1ea35b2f954170d1a', 'training/backbone_benchmark/test_benchmark.py': '0b6920f94afe65106522c21afce2494c820b9e0a515284cb3c1ebb26f526316e', 'training/component_ablation/features.py': 'd3dd0f1fcbf2dea61a1ee5070c82e046cca4f1d75fdbccd6badd77321d7e5680', 'training/component_ablation/models.py': 'a43c0ea2a85caa2aaf890dc81ac7cf5f67d2237eeba5614fd00492b89285583f', 'training/component_ablation/training.py': '1a9fb95e0a29a2575c7e75aa541378f0e8918094a8cfe556faca144b2dc2d05f', 'training/component_ablation/validate_pipeline.py': 'a60a05359eda846c1a777510011a068cbeaad185413445c86d9441f0c8d7a8a6'}
CODE_ROOT = Path('/kaggle/working/a3-tuning-source-' + SOURCE_COMMIT[:7])
RUN_ROOT = Path('/kaggle/working/a3-tuning-20261121')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
def fetch_source(item):
    name, expected = item
    target = CODE_ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        content = target.read_bytes()
    else:
        url = 'https://raw.githubusercontent.com/aBaoN9/me-stgcn-ta-fall-detection/' + SOURCE_COMMIT + '/' + name
        with urllib.request.urlopen(url, timeout=60) as response:
            content = response.read()
    assert hashlib.sha256(content).hexdigest() == expected, 'Source SHA mismatch: ' + name
    if not target.exists():
        target.write_bytes(content)
with ThreadPoolExecutor(max_workers=6) as pool:
    list(pool.map(fetch_source, SOURCE_MANIFEST.items()))
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
launch = dict(source_commit=SOURCE_COMMIT, source_sha256=SOURCE_MANIFEST, python=platform.python_version(), torch=torch.__version__, gpus=[torch.cuda.get_device_name(i) for i in range(2)], started_unix=time.time())
(RUN_ROOT / 'launch_manifest.json').write_text(json.dumps(launch, indent=2))
print('SOURCE_VERIFIED', SOURCE_COMMIT, len(SOURCE_MANIFEST), 'files', flush=True)
print('GPU_READY', launch['gpus'], launch['python'], launch['torch'], flush=True)


In [ ]:
import subprocess

def run_logged(name, arguments):
    started = time.monotonic()
    with (RUN_ROOT / (name + '.log')).open('a', encoding='utf-8') as stream:
        process = subprocess.Popen(arguments, cwd=CODE_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                stream.write(line)
                stream.flush()
                print(line.rstrip()[:1400], flush=True)
            code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
    print(name, 'EXIT', code, 'SECONDS', round(time.monotonic() - started, 2), flush=True)
    if code:
        raise RuntimeError(name + ' failed; inspect saved log, no full run allowed')

run_logged('input_verification', [sys.executable, '-B', 'scripts/training_inputs.py', '--input-root', 'data/gmdcsa24', '--download'])
tests = "import unittest, torch; torch.set_num_threads(2); suite=unittest.defaultTestLoader.loadTestsFromNames(['training.a3_tuning.test_runner','training.backbone_benchmark.test_benchmark','training.backbone_benchmark.supplement.test_supplement','training.backbone_benchmark.supplement.tools.test_monitor_windows']); result=unittest.TextTestRunner(verbosity=1).run(suite); raise SystemExit(not result.wasSuccessful())"
run_logged('regression_tests', [sys.executable, '-B', '-c', tests])


In [ ]:
run_logged('gpu_smoke', [sys.executable, '-u', '-B', '-m', 'training.a3_tuning.runner', '--mode', 'smoke', '--device', 'cuda', '--input-root', 'data/gmdcsa24', '--output', str(RUN_ROOT / 'smoke')])
smoke_record = json.loads((RUN_ROOT / 'smoke/smoke.json').read_text())
assert smoke_record['status'] == 'SMOKE_PASS' and smoke_record['fits'] == 12
print('REAL_GPU_SMOKE_PASS: 12 fits; full may start', flush=True)


In [ ]:
run_logged('full_training', [sys.executable, '-u', '-B', '-m', 'training.a3_tuning.runner', '--mode', 'full', '--device', 'cuda', '--workers', '2', '--input-root', 'data/gmdcsa24', '--smoke-evidence', str(RUN_ROOT / 'smoke'), '--output', str(RUN_ROOT / 'full')])
completion = json.loads((RUN_ROOT / 'full/completion.json').read_text())
assert completion['status'] == 'FULL_COMPLETE' and completion['completed_fits'] == 684
print('FULL_TRAINING_COMPLETE; independent review and latency still required', flush=True)
import pandas as pd
display(pd.read_csv(RUN_ROOT / 'full/pooled_by_seed.csv'))


In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/a3-tuning-20261121-result', 'gztar', root_dir='/kaggle/working', base_dir='a3-tuning-20261121')
print('RESULT_ARCHIVE', archive, flush=True)
